# 0921 注意力对照实验：加性（Bahdanau）与乘性（投影点积）

> 实验日期：2026-09-21 ｜ 数据：Tatoeba cmn.txt（沿用 0918 划分，seed=10）｜ 设备：单卡 CUDA

起点是 0918 的无注意力 Seq2Seq（val CE 2.8050 / BLEU-4 0.1297）。今天在它基础上实现注意力，并做了两个版本：

| 版本 | 位置 | 注意力形式 |
| --- | --- | --- |
| `0921/` | GRU **之前**（用上一步状态 $s_{t-1}$ 去问） | Bahdanau **加性**：$v^\top \tanh(W_1 h_s + W_2 s_{t-1})$ |
| `teacher0921/` | GRU **之后**（用当前状态 $s_t$ 查询） | **乘性（投影点积 / 双线性）**：$s_t^\top W_a h_s$ |

## 架构对比

| 对比项 | 0921 加性版 | teacher0921 乘性版 |
| --- | --- | --- |
| 打分参数 | `W1/W2/v` 三个可学习投影 | `encoder_project` 是可学习的 $W_a$（2H→H）；点积运算本身无参数 |
| 打分时机 | GRU 前：用 `hidden[-1]` | GRU 后：用当前 `output`（= hn[-1]） |
| context 去向 | 与词向量拼接 → 喂进 GRU 输入 | 与 hn 拼接 → `combine_project` → `fc` |
| Decoder 隐维 | 128（= 2×64） | 64（Encoder 端 `fc(2H→H)` 降维） |
| 打分计算 | `v(tanh(W1(enc) + W2(hidden).unsqueeze(1)))` | `bmm(dec_out, encoder_project(enc).transpose(1,2))` |
| PAD 掩码 | `masked_fill(==0, -1e9)` | 同款 |
| 输出形状 | `[B, tgt_len-1, V]` | `[B, tgt_len, V]`（第 0 行留零，训练侧切 `[:,1:]`） |
| 注意力可视化 | `greedy_decode(return_attention=True)` | 同款移植+ `attention_visualize.py` |

## v1 训练结果（历史记录，配置未统一）

### 配置对照

| 配置 | 0921 加性版 | teacher0921 乘性版 |
| --- | --- | --- |
| batch size | 64 | 16 |
| 计划轮数 / TF 衰减 | 50 轮，TF 1.0→0.5 | 70 轮，TF 1.0→0.5 |
| 早停 | epoch 32（best=24） | epoch 15（best=7） |
| 停止时 TF | ≈0.69（best 轮时 ≈0.77） | ≈0.90（best 轮时 ≈0.96） |
| 训练耗时 | 约 13 分钟 | 约 30 分钟 |

### 指标对照（全量验证集 2639 句）

| 版本 | val CE（teacher forcing 口径） | BLEU-4 | 相对无注意力基线 |
| --- | --- | --- | --- |
| 0918 无注意力（参照） | 2.8050 | 0.1297 | — |
| **0921 加性版** | 2.8047 | **0.1389** | **+7.1%** |
| **teacher0921 乘性版** | **2.7147** | 0.1307 | +0.8% |

### 解读：条件预测能力与自由生成质量并不等价

- **加性版**：CE 与基线接近（2.8047 vs 2.8050），BLEU 相对提高约7.1%。本次运行中，自由生成指标的改善比 CE 更明显。
- **乘性版**：CE 较基线低约0.09，BLEU 相对提高约0.8%。给定正确前缀时的预测改善，未对应同等幅度的自由生成提升。

CE 使用完整 Teacher Forcing；BLEU 使用模型自行生成的前缀。两者衡量不同条件下的表现。下文的热力图和训练计划提供分析线索，但不足以单独确定原因。

## v1 热力图观察（历史示例：汤姆不想让玛丽失望。）

以下数值来自 v1 历史记录，用于观察该句的注意力分布，不能代表全验证集的对齐质量。下方 v1 热力图由归档的 v1 权重重新生成（`attention_heatmap_v1.png`，贪心解码是确定性的，与原图一致），不会再被覆盖。

**0921 加性版：权重较分散，逐词对应关系不清晰**

- `Tom` 最大权重落在"不想"（0.60），未呈现直接的 Tom—汤姆对应；
- `Mary` 的权重分布在多个位置（不想 0.22 / 失望 0.22 / 玛丽 0.13 / 汤姆 0.11）；分布较散本身不能证明信息被保留或丢失；
- `does n't / want / to` 明显偏向句尾"。"与 `<EOS>`。

**teacher0921 乘性版：该示例的权重高度集中于句尾**

- 汤姆 / 不想 / 让 / 玛丽 / 失望 全部 ≈ 0.00；
- `Tom → <EOS> 0.96`、`to → <EOS> 0.95`、`. → <EOS> 0.99`；
- 该模式提示模型可能较多依赖句尾摘要，而未充分利用动态源位置选择。双向 GRU 的各位置表示包含上下文，句尾状态也包含源句信息，因此不能解释为"根本没读源句"，也不能仅凭该图认定其为 BLEU 偏低的直接原因。

### v1 热力图

上图为 0921 加性版 v1（权重较分散）；下图为 teacher0921 乘性版 v1（权重高度集中于句尾）。图片由归档的 v1 权重重新生成并另存为 `attention_heatmap_v1.png`，与本节文字记录一一对应，不受后续重跑影响。

![0921 加性版 v1 热力图](0921/attention_maps/attention_heatmap_v1.png)

![teacher0921 乘性版 v1 热力图](teacher0921/attention_maps/attention_heatmap_v1.png)

## v1 结果解读与待验证假设

1. **注意力分布**：v1 示例中，乘性版权重集中于句尾，加性版相对分散。乘性版已有可学习投影 $W_a$；点积分数受向量模长和方向共同影响。分数尺度与分布集中的关系仍需验证，不能将其归因为“无参数”，也不能认为加性结构必然避免集中。
2. **训练计划差异**：乘性版按70轮衰减 TF，但在第15轮早停，最佳第7轮的 TF 约0.96；加性版最佳第24轮的 TF 约0.77。接触自身预测前缀的比例不同，可能影响自由生成表现，但当前实验没有隔离该因素。
3. **公平性 caveat**：两版的 batch（64 vs 16）、TF 计划（50 vs 70 轮）、有效轮数（24 vs 7）、架构均不同，且各只跑了单 seed —— 严格表述是"**乘性点积版在当前训练计划下未超过加性版**"，而不是"乘性注意力一定更差"。


## v2 对照结果（统一主要训练超参数）

> 上方第 1~5 格是第一轮（v1）的记录，热力图为 v1 模型生成并已直接嵌入；本节为统一配方重训后的结果，热力图见下方嵌图。

### 为什么重赛

v1 两版配置不同：乘性版 batch=16，每轮优化器更新次数约为 batch=64 的4倍；计划70轮，在第15轮早停，最佳第7轮的 TF 约0.96。这些差异限制了比较，因此 v2 统一主要训练超参数后重新训练。

### 统一的训练超参数

| 项目 | 统一值 |
| --- | --- |
| batch size | 64 |
| 轮数 / TF 衰减 | 50 轮，1.0 → 0.5 |
| 优化器 | AdamW，lr 0.001，weight_decay 1e-4 |
| 学习率调度（新增） | ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5) |
| 早停耐心 | 10 |
| clip / dropout / seed | 1.0 / 0.3 / 10 |

### 成绩单（全量验证集 2639 句）

| 版本 | 配置 | val CE | BLEU-4 |
| --- | --- | --- | --- |
| 0918 无注意力（参照） | — | 2.8050 | 0.1297 |
| 0921 加性 v1 | 旧配置 | 2.8047 | 0.1389 |
| teacher 乘性 v1 | 旧配置 | 2.7147 | 0.1307 |
| **0921 加性 v2** | 统一配方 | 2.7976 | **0.1526** |
| **teacher 乘性 v2** | 统一配方 | 2.7419 | **0.1746** |

### 三个发现

1. **本次结果对训练配置较敏感**：重新训练后，加性版 BLEU 从0.1389提高到0.1526（约+10%），乘性版从0.1307提高到0.1746（约+34%）。多项设置同时改变，尚不能将提升归因于某一项，也不能据此判断配置与架构的一般重要性排序。
2. **本次乘性实现表现更好**：按未四舍五入的评估值，乘性版 BLEU 为0.174644，加性版为0.152565，相差约2.21个百分制 BLEU 点，相对高约14.5%；乘性版 CE 也更低。BLEU 不是翻译正确率。
3. **该示例的部分语义对齐改善**：乘性版 v2 的 Tom→汤姆、does / n't / want→不想、首个 Mary→玛丽权重较集中，后半句仍重复生成 Mary；加性版该句分布较散。该观察说明分布可能随训练变化，但不足以确认所谓“塌缩”的成因或全局改善程度。

> 结论边界：两版验证集和主要训练超参数一致，但 Decoder 隐维、context 维度、Attention 插入位置及参数量仍不同。这是两套完整实现的单 seed 对照，结果尚不能单独归因于打分公式；优势的稳定性需要多 seed 复验。

### v2 热力图

同一句话（汤姆不想让玛丽失望。）下，上图加性版的逐词对应较不清晰；下图乘性版前半句出现合理对齐：Tom→汤姆约1.00、首个 Mary→玛丽约0.92、does / n't / want→不想约1.00。数值保留两位小数，1.00不代表精确等于1。

乘性版仍输出 `Tom does n't want Mary to Mary .`：第二个 Mary 的权重主要落在“失望”（约0.70）。关注到相关源位置，不保证输出层生成正确词语。加性版输出 `Tom does n't want to to dance .`，也存在重复和语义错误。

双向 GRU 的源位置表示包含上下文；热力图展示权重分配，不能直接等同于模型完整推理过程。以上为单句观察，不代表全验证集的对齐质量。

![0921 加性版 v2 热力图](0921/attention_maps/attention_heatmap_v2.png)

![teacher0921 乘性版 v2 热力图](teacher0921/attention_maps/attention_heatmap_v2.png)


In [1]:
# 加性注意力的实现（Bahdanau）：e = vᵀ tanh(W1·h_s + W2·s)
import torch
import torch.nn as nn


class AddAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attn_dim):
        super().__init__()
        self.W1 = nn.Linear(encoder_dim, attn_dim, bias=False)  # 投影编码器各位置输出
        self.W2 = nn.Linear(decoder_dim, attn_dim, bias=False)  # 投影解码器状态（查询）
        self.v = nn.Linear(attn_dim, 1, bias=False)             # 对齐空间 → 每个位置一个分数

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        # decoder_hidden:  [B, decoder_dim]      解码器状态（去问）
        # encoder_outputs: [B, L, encoder_dim]   编码器各位置输出（被查）
        # mask:            [B, L]，True=真实词（可选，屏蔽 PAD）
        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(decoder_hidden).unsqueeze(1))
        energy = self.v(energy).squeeze(-1)                 # [B, L] 注意力分数
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e9)    # PAD 不参与 softmax
        attn_weights = torch.softmax(energy, dim=-1)        # [B, L]
        # context：按权重对编码器输出加权求和（value 用原始 encoder_outputs）
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attn_weights


# 快速验证：形状 + 行和 + PAD 位置权重
torch.manual_seed(0)
attn = AddAttention(encoder_dim=128, decoder_dim=128, attn_dim=64)
decoder_hidden = torch.randn(3, 128)
encoder_outputs = torch.randn(3, 7, 128)
mask = torch.tensor([[1, 1, 1, 1, 1, 0, 0],
                     [1, 1, 1, 0, 0, 0, 0],
                     [1, 1, 1, 1, 1, 1, 1]], dtype=torch.bool)
context, attn_weights = attn(decoder_hidden, encoder_outputs, mask)
print("context:", tuple(context.shape), "| attn:", tuple(attn_weights.shape),
      "| 行和:", [round(x, 4) for x in attn_weights.sum(-1).tolist()])
print("PAD 位置权重（前两行应为 0）:", attn_weights[:, 5:7].tolist())

context: (3, 128) | attn: (3, 7) | 行和: [1.0, 1.0, 1.0]
PAD 位置权重（前两行应为 0）: [[0.0, 0.0], [0.0, 0.0], [0.15773527324199677, 0.1523521989583969]]


In [2]:
# 乘性注意力的实现（投影点积 / Luong）：e = sᵀ W_a h_s
import torch
import torch.nn as nn


class MultiAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim):
        super().__init__()
        # W_a：把编码器输出投影到与解码器状态同维，点积才能进行（也可理解为学习一个"键空间"）
        self.W_a = nn.Linear(encoder_dim, decoder_dim, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        # decoder_hidden:  [B, decoder_dim]      解码器状态（查询）
        # encoder_outputs: [B, L, encoder_dim]   编码器各位置输出
        # mask:            [B, L]，True=真实词（可选，屏蔽 PAD）
        projected = self.W_a(encoder_outputs)                                   # [B, L, decoder_dim]
        # 点积打分：查询 · 投影后的键，需把键转置成 [B, decoder_dim, L]
        scores = torch.bmm(decoder_hidden.unsqueeze(1), projected.transpose(1, 2)).squeeze(1)  # [B, L]
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn_weights = torch.softmax(scores, dim=-1)                            # [B, L]
        # context：按权重对投影后的编码器输出加权求和（与 teacher0921 实现一致）
        context = torch.bmm(attn_weights.unsqueeze(1), projected).squeeze(1)
        return context, attn_weights


# 快速验证：形状 + 行和 + PAD 位置权重
torch.manual_seed(0)
attn = MultiAttention(encoder_dim=128, decoder_dim=64)
decoder_hidden = torch.randn(3, 64)
encoder_outputs = torch.randn(3, 7, 128)
mask = torch.tensor([[1, 1, 1, 1, 1, 0, 0],
                     [1, 1, 1, 0, 0, 0, 0],
                     [1, 1, 1, 1, 1, 1, 1]], dtype=torch.bool)
context, attn_weights = attn(decoder_hidden, encoder_outputs, mask)
print("context:", tuple(context.shape), "| attn:", tuple(attn_weights.shape),
      "| 行和:", [round(x, 4) for x in attn_weights.sum(-1).tolist()])
print("PAD 位置权重（前两行应为 0）:", attn_weights[:, 5:7].tolist())

context: (3, 64) | attn: (3, 7) | 行和: [1.0, 1.0, 1.0]
PAD 位置权重（前两行应为 0）: [[0.0, 0.0], [0.0, 0.0], [7.736307150452149e-10, 0.00895063765347004]]
